# Exploration du marché data France

Notebook d'exploration sur l'état actuel de `fct_offre` et des marts associés
(552 offres, extraction du 17/07/2026). Pas d'historique temporel disponible —
`fct_marche_hebdo` n'existe pas encore — donc cette exploration porte sur un
instantané, pas une évolution.

In [1]:
import duckdb

con = duckdb.connect("../data/warehouse.duckdb", read_only=True)

# Vérification rapide : la base est-elle accessible et fct_offre à jour ?
con.execute("select count(*) from fct_offre").fetchone()

(552,)

## 1. Vue d'ensemble

In [2]:
con.execute("""
select
    type_contrat,
    count(offre_id) as nb_offres,
    round((count(offre_id) / sum(count(offre_id)) over ()) * 100, 2) as pc_offres
from fct_offre
group by type_contrat
order by nb_offres desc""").df()

,type_contrat,nb_offres,pc_offres
0,CDI,395,71.56
1,CDD,89,16.12
2,MIS,58,10.51
3,LIB,9,1.63
4,CCE,1,0.18


CDI et CDD concentrent 484/552 offres (87,7 %). `CCE` n'a qu'une seule offre —
à regrouper avec les catégories rares si ce champ est graphé plus tard (un
camembert à 5 tranches dont une à 0,2 % n'est pas lisible).

In [3]:
con.execute("""
select
    categorie_employeur, 
    count(offre_id) as nb_offres,
    round((count(offre_id) / sum(count(offre_id)) over ()) * 100.0, 2) as pc_offres
from fct_offre
group by categorie_employeur
order by nb_offres desc
""").df()

,categorie_employeur,nb_offres,pc_offres
0,EMPLOYEUR_DIRECT,213,38.59
1,ANONYME,187,33.88
2,INTERMEDIAIRE,131,23.73
3,INTERMEDIAIRE_reclasse,21,3.80


## 2. Compétences demandées

`fct_offre_technologie` et `fct_offre_domaine` ont un grain plus fin que
`fct_offre` : une ligne = un couple (offre, terme). `count(distinct offre_id)`
systématique pour éviter tout surcomptage.

In [4]:
con.execute("""
select
    technologie,
    count(distinct offre_id) as nb_offres
from fct_offre_technologie
group by technologie
order by nb_offres desc
limit 10
""").df()

,technologie,nb_offres
0,Python,152
1,SQL,148
2,Power BI,104
3,AWS,32
4,Databricks,31
5,Git,27
6,Azure,26
7,Tableau,26
8,Snowflake,25
9,Spark,24


In [5]:
con.execute("""
select 
    domaine_normalise,
    count(distinct offre_id) as nb_offres,
    round(count(distinct offre_id) * 100.0 / (select count(*) from fct_offre), 2) as pc_offres
from fct_offre_domaine
group by domaine_normalise
order by nb_offres desc
limit 5
""").df()

,domaine_normalise,nb_offres,pc_offres
0,Gouvernance des données,122,22.10
1,Analyse de données,109,19.75
2,Qualité des données,68,12.32
3,Business Intelligence,67,12.14
4,Machine Learning,66,11.96


## 3. Rémunération

Restreint aux salaires `salaire_periode = 'annuel'` (152 offres). Les salaires
horaires et mensuels restent hors de toute analyse statistique — échantillon
trop faible pour être défendable (19 et 1 offres, cf. `assert_bornes_salaire_annuel`
et README, limite vérifiée sans changement en Session 7).

In [6]:
con.execute("""
select
    min(salaire_min),
    max(salaire_min),
    avg(salaire_min),
    median(salaire_min)
from fct_offre
where salaire_periode = 'annuel'
""").df()

,min(salaire_min),max(salaire_min),avg(salaire_min),median(salaire_min)
0,15,100000,43955.506579,42000.0


In [7]:
con.execute("""
select 
    count(offre_id)
from fct_offre
where salaire_min < 1000
""").df()

,count(offre_id)
0,4


In [8]:
con.execute("""
select
    offre_id
from fct_offre
where salaire_min < 1000
""").df()

,offre_id
0,211BQXG
1,5023662
2,211FNNV
3,4933945


In [9]:
con.execute("""
select
    salaire_libelle
from stg_ft_offres
where offre_id in ('211BQXG', '5023662', '211FNNV', '4933945')
""").df()

,salaire_libelle
0,Annuel de 15.0 Euros
1,Mensuel de 802.82 Euros à 989.52 Euros
2,Mensuel de 989.51 Euros à 1138.87 Euros sur 12...
3,Horaire de 25.0 Euros sur 12 mois


In [10]:
con.execute("""
select 
    offre_id
from fct_offre
where salaire_min < 1000 and salaire_periode = 'annuel'
""").df()

,offre_id
0,4933945


In [11]:
con.execute("""
select
    offre_id
from fct_offre
where salaire_min = 100000
""").df()

,offre_id
0,1620026


In [12]:
con.execute("""
select
    salaire_libelle,
    description
from stg_ft_offres
where offre_id = '1620026'
""").df()

,salaire_libelle,description
0,Annuel de 100000.0 Euros à 110000.0 Euros,Lead Solution Architect H/F – CDI – Paris – Ar...


`min(salaire_min) = 15` sur le premier passage : `salaire_libelle` = "Annuel de
15.0 Euros" (offre 4933945) — coquille de saisie confirmée sur le texte source,
pas une erreur de parsing du pipeline (cf. limite assumée README). Exclue des
statistiques ci-dessous via `offre_id != '4933945'`.

`max(salaire_min) = 100000` vérifié à part : "Lead Solution Architect", plage
100k-110k€, salaire légitime — conservé.

In [13]:
con.execute("""
select
    min(salaire_min),
    max(salaire_min),
    avg(salaire_min),
    median(salaire_min)
from fct_offre
where salaire_periode = 'annuel' and offre_id != '4933945'
""").df()

,min(salaire_min),max(salaire_min),avg(salaire_min),median(salaire_min)
0,25000,100000,44246.503311,42000.0


## 4. Géographie

In [20]:
con.execute("""
select
    c.commune,
    c.code_postal,
    count(distinct o.offre_id) as nb_offres
from fct_offre o
join dim_commune c on o.code_postal = c.code_postal
group by c.commune, c.code_postal
""").df()
    

,commune,code_postal,nb_offres
0,06065,06610,1
1,77288,77000,2
2,59350,59000,14
3,75103,75003,1
4,92046,92240,1
...,...,...,...
188,92020,92320,1
189,59346,59260,1
190,73064,73190,1
191,73065,73000,1


In [19]:
con.execute("""
select
    code_postal,
    count(distinct offre_id) as nb_offres
from fct_offre
group by code_postal
order by nb_offres desc
limit 10
""").df()

,code_postal,nb_offres
0,None,107
1,75009,16
2,92400,15
3,31000,15
4,59000,14
5,92000,13
6,44000,13
7,31700,11
8,33000,10
9,92800,8


In [21]:
con.execute("DESCRIBE dim_commune").df()

,column_name,column_type,null,key,default,extra
0,code_postal,VARCHAR,YES,None,None,None
1,commune,VARCHAR,YES,None,None,None


In [23]:
codes = con.execute("""
    select distinct code_postal
    from fct_offre
    where code_postal is not null
""").df()['code_postal'].tolist()
print(','.join(codes))

75017,75015,69300,62300,78180,77310,91300,93110,78140,13001,78000,94200,92160,78960,33700,64240,69760,09100,31000,78200,06410,94260,03000,59175,44240,67300,70000,94800,75009,57000,46100,59491,54000,69003,38000,75003,92110,94110,40000,13600,06150,92240,74210,95000,85500,11800,78990,73200,44000,33160,69002,13107,92130,91190,75010,67290,92000,54600,92500,06510,75013,77150,92210,94600,92350,27000,31700,01300,92250,93160,35170,94150,95130,31770,69100,67230,91810,51520,10400,29490,36200,94410,69007,69001,34000,59100,33600,35510,75008,93260,77600,13700,39130,13790,78300,92340,75002,50260,25290,73000,92400,69005,13009,92200,92320,62138,91130,92100,45160,75014,78370,67600,92390,72000,91000,73190,78280,86360,81000,01100,66000,45340,42000,79000,51100,69800,95350,69009,21000,83000,77000,93410,59260,13080,59170,94490,56250,92300,30000,64510,13122,82160,37420,94000,33000,44300,57300,93380,55300,59000,66530,94270,84000,67000,53000,77184,92800,69190,75001,78860,81310,16000,45000,56100,56000,92120,6300